In [1]:
import pandas as pd
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive
from src.analysis.poissonFunctions import (
    compute_bayesian_lambda,
    compute_bayesian_lambda_assists,
    compute_bayesian_lambda_rebounds,
    compute_bayesian_lambda_blocks,
    compute_bayesian_lambda_steals
)

from scipy.stats import poisson
import numpy as np
from nba_api.stats.endpoints import leaguedashteamstats

In [2]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
s26.rename(columns={'BLK_x': 'BLK'}, inplace=True)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_DFS_20251211_164531.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,Anfernee Simons,Over,11.5,-137,2025-12-12,2025-12-12T00:44:53Z,2025-12-11 16:45:31
1,Underdog,player_points,Anfernee Simons,Under,11.5,-137,2025-12-12,2025-12-12T00:44:53Z,2025-12-11 16:45:31
2,Underdog,player_points,Jaylen Brown,Over,29.5,-137,2025-12-12,2025-12-12T00:44:53Z,2025-12-11 16:45:31
3,Underdog,player_points,Jaylen Brown,Under,29.5,-137,2025-12-12,2025-12-12T00:44:53Z,2025-12-11 16:45:31
4,Underdog,player_points,Myles Turner,Over,12.5,-137,2025-12-12,2025-12-12T00:44:53Z,2025-12-11 16:45:31


## Points

### prizepicks

In [3]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]
res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_off_rtg = league_df['OFF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_pts = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_off_rtg, league_avg_def_rtg,
        league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_pts % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_pts) + 1)
    elif target_pts % 1 == 0:
        prob_over_poisson = poisson.sf(target_pts, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_pts), int(target_pts) + 1)
    else:
        # Handle other cases
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_pts,
        'L-5': round(count_line_hits(player_df, target_pts, 'player_points', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_pts, 'player_points', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_pts, 'player_points', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })

point_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
point_df.to_csv(f'data/props/prizepicks/player_points.csv', index=False)
point_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Devin Vassell,11.5,0.6,0.8,0.67,0.942,0.058
1,De'Anthony Melton,8.5,0.4,0.2,0.13,0.879,0.121
2,Jordan Poole,12.5,0.4,0.4,0.27,0.866,0.134
3,Aaron Holiday,8.5,0.8,0.8,0.60,0.864,0.136
4,Stephen Curry,24.5,0.6,0.6,0.60,0.847,0.153
5,Harrison Barnes,10.5,0.8,0.7,0.67,0.831,0.169
6,Kawhi Leonard,22.5,0.6,0.6,0.53,0.813,0.187
7,Jordan Walsh,7.5,1.0,0.6,0.47,0.807,0.193
8,Chet Holmgren,16.5,0.8,0.6,0.67,0.790,0.210
9,Rudy Gobert,9.5,0.6,0.6,0.67,0.789,0.211


### underdog

In [4]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]
res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_off_rtg = league_df['OFF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_pts = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_off_rtg, league_avg_def_rtg,
        league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_pts % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_pts) + 1)
    elif target_pts % 1 == 0:
        prob_over_poisson = poisson.sf(target_pts, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_pts), int(target_pts) + 1)
    else:
        # Handle other cases
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_pts,
        'L-5': round(count_line_hits(player_df, target_pts, 'player_points', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_pts, 'player_points', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_pts, 'player_points', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })

point_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
point_df.to_csv(f'data/props/underdog/player_points.csv', index=False)
point_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,De'Anthony Melton,8.5,0.4,0.2,0.13,0.879,0.121
1,Stephen Curry,24.5,0.6,0.6,0.60,0.847,0.153
2,Kawhi Leonard,22.5,0.6,0.6,0.53,0.813,0.187
3,Chet Holmgren,16.5,0.8,0.6,0.67,0.790,0.210
4,Jabari Walker,4.5,0.6,0.5,0.40,0.789,0.211
5,Dominick Barlow,6.5,0.6,0.6,0.60,0.784,0.216
6,Jamal Murray,24.5,0.4,0.5,0.40,0.775,0.225
7,Quinten Post,7.5,0.8,0.7,0.53,0.774,0.226
8,T.J. McConnell,6.5,0.4,0.7,0.60,0.755,0.245
9,Saddiq Bey,16.5,1.0,0.9,0.67,0.741,0.259


## Assists

### prizepicks

In [5]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_assists')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_ast = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_assists(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_def_rtg, league_avg_pace,
        league_avg_ast_ratio, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_ast % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_ast) + 1)
    elif target_ast % 1 == 0:
        prob_over_poisson = poisson.sf(target_ast, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_ast), int(target_ast) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_ast,
        'L-5': round(count_line_hits(player_df, target_ast, 'player_assists', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_ast, 'player_assists', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_ast, 'player_assists', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
assist_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
assist_df.to_csv(f'data/props/prizepicks/player_assists.csv', index=False)
assist_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Trey Murphy III,3.5,1.0,0.7,0.60,0.664,0.336
1,Rudy Gobert,1.5,0.8,0.8,0.60,0.648,0.352
2,De'Anthony Melton,1.5,0.2,0.1,0.07,0.638,0.362
3,Jaylen Brown,5.0,0.6,0.4,0.33,0.619,0.381
4,Julius Randle,5.0,0.8,0.6,0.53,0.616,0.384
5,Buddy Hield,1.5,0.6,0.5,0.53,0.590,0.410
6,Jalen Williams,5.5,0.4,0.3,0.20,0.570,0.430
7,Derik Queen,4.5,0.8,0.6,0.53,0.560,0.440
8,James Harden,7.5,0.4,0.5,0.53,0.543,0.457
9,Desmond Bane,4.5,0.6,0.7,0.67,0.542,0.458


### underdog

In [6]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_assists')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_ast = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_assists(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_def_rtg, league_avg_pace,
        league_avg_ast_ratio, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_ast % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_ast) + 1)
    elif target_ast % 1 == 0:
        prob_over_poisson = poisson.sf(target_ast, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_ast), int(target_ast) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_ast,
        'L-5': round(count_line_hits(player_df, target_ast, 'player_assists', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_ast, 'player_assists', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_ast, 'player_assists', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
assist_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
assist_df.to_csv(f'data/props/underdog/player_assists.csv', index=False)
assist_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Trey Murphy III,3.5,1.0,0.7,0.60,0.664,0.336
1,Jose Alvarado,2.5,0.6,0.6,0.53,0.607,0.393
2,Buddy Hield,1.5,0.6,0.5,0.53,0.590,0.410
3,Bobby Portis,1.5,0.4,0.6,0.40,0.563,0.437
4,Jalen Suggs,4.5,0.2,0.4,0.40,0.509,0.491
5,VJ Edgecombe,3.5,0.4,0.4,0.47,0.503,0.497
6,Cameron Johnson,2.5,0.6,0.5,0.53,0.491,0.509
7,Bennedict Mathurin,2.5,0.4,0.4,0.33,0.467,0.533
8,Aaron Holiday,1.5,0.6,0.5,0.47,0.463,0.537
9,Josh Hart,5.5,0.4,0.6,0.60,0.417,0.583


# REBOUNDS

### prizepicks

In [7]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_rebounds')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
league_avg_dreb = league_df['DREB_PCT'].mean()
league_avg_reb = league_df['REB_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_reb = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_rebounds(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_reb,
        league_avg_oreb, league_avg_dreb, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_reb % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_reb) + 1)
    elif target_reb % 1 == 0:
        prob_over_poisson = poisson.sf(target_reb, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_reb), int(target_reb) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_reb,
        'L-5': round(count_line_hits(player_df, target_reb, 'player_rebounds', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_reb, 'player_rebounds', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_reb, 'player_rebounds', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
rebound_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
rebound_df.to_csv(f'data/props/prizepicks/player_rebounds.csv', index=False)
rebound_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Amen Thompson,6.5,0.6,0.8,0.73,0.754,0.246
1,Yves Missi,4.5,0.8,0.6,0.60,0.696,0.304
2,Duncan Robinson,2.5,0.4,0.6,0.47,0.695,0.305
3,Ausar Thompson,5.5,0.8,0.6,0.60,0.652,0.348
4,Jaylen Brown,6.5,0.6,0.6,0.47,0.643,0.357
5,Lonzo Ball,4.5,1.0,0.7,0.60,0.626,0.374
6,Evan Mobley,9.5,0.8,0.7,0.60,0.619,0.381
7,Kevin Durant,4.5,0.4,0.6,0.47,0.604,0.396
8,Jalen Johnson,10.5,1.0,0.7,0.53,0.594,0.406
9,Spencer Jones,3.5,0.6,0.6,0.40,0.561,0.439


### underdog

In [8]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_rebounds')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
league_avg_dreb = league_df['DREB_PCT'].mean()
league_avg_reb = league_df['REB_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_reb = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_rebounds(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_reb,
        league_avg_oreb, league_avg_dreb, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_reb % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_reb) + 1)
    elif target_reb % 1 == 0:
        prob_over_poisson = poisson.sf(target_reb, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_reb), int(target_reb) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_reb,
        'L-5': round(count_line_hits(player_df, target_reb, 'player_rebounds', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_reb, 'player_rebounds', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_reb, 'player_rebounds', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
rebound_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
rebound_df.to_csv(f'data/props/underdog/player_rebounds.csv', index=False)
rebound_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Jarace Walker,3.5,0.2,0.6,0.53,0.609,0.391
1,Spencer Jones,3.5,0.6,0.6,0.40,0.561,0.439
2,Kris Dunn,2.5,0.8,0.7,0.53,0.555,0.445
3,Ivica Zubac,10.5,0.8,0.7,0.73,0.553,0.447
4,Nickeil Alexander-Walker,3.5,0.8,0.7,0.53,0.550,0.450
5,Derrick White,4.5,0.4,0.6,0.53,0.543,0.457
6,Jalen Duren,11.5,0.4,0.5,0.53,0.537,0.463
7,Precious Achiuwa,5.5,0.6,0.5,0.40,0.537,0.463
8,Josh Giddey,8.5,0.6,0.6,0.67,0.532,0.468
9,De'Aaron Fox,3.5,1.0,0.8,0.73,0.521,0.479


## Blocks

### prizepicks

In [9]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_blocks')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_blk = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_blocks(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_blk % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_blk) + 1)
    elif target_blk % 1 == 0:
        prob_over_poisson = poisson.sf(target_blk, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_blk), int(target_blk) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_blk,
        'L-5': round(count_line_hits(player_df, target_blk, 'player_blocks', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_blk, 'player_blocks', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_blk, 'player_blocks', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
blocks_df = pd.DataFrame(res)
blocks_df.to_csv(f'data/props/prizepicks/player_blocks.csv', index=False)
blocks_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%,IMPLIED_ODDS
0,John Collins,0.5,0.4,0.5,0.47,0.541,0.459,1.847
1,Deni Avdija,0.5,0.2,0.4,0.40,0.359,0.641,2.785
2,Evan Mobley,1.5,0.6,0.6,0.47,0.548,0.452,1.825
3,Anthony Edwards,0.5,0.6,0.7,0.60,0.627,0.373,1.594
4,Quinten Post,0.5,0.6,0.5,0.47,0.509,0.491,1.964
5,Jalen Suggs,0.5,0.6,0.7,0.60,0.582,0.418,1.718
6,Victor Wembanyama,2.5,0.6,0.6,0.53,0.625,0.375,1.601
7,Shai Gilgeous-Alexander,0.5,0.4,0.4,0.40,0.349,0.651,2.865


### underdog

In [10]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_blocks')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_blk = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_blocks(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_blk % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_blk) + 1)
    elif target_blk % 1 == 0:
        prob_over_poisson = poisson.sf(target_blk, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_blk), int(target_blk) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_blk,
        'L-5': round(count_line_hits(player_df, target_blk, 'player_blocks', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_blk, 'player_blocks', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_blk, 'player_blocks', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
blocks_df = pd.DataFrame(res)
blocks_df.to_csv(f'data/props/underdog/player_blocks.csv', index=False)
blocks_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%,IMPLIED_ODDS
0,Matas Buzelis,1.5,0.4,0.6,0.60,0.451,0.549,2.220
1,Victor Wembanyama,2.5,0.6,0.6,0.53,0.625,0.375,1.601


# STEALS

### prizepicks

In [11]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_steals')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_stl = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_steals(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_stl % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_stl) + 1)
    elif target_stl % 1 == 0:
        prob_over_poisson = poisson.sf(target_stl, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_stl), int(target_stl) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_stl,
        'L-5': round(count_line_hits(player_df, target_stl, 'player_steals', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_stl, 'player_steals', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_stl, 'player_steals', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
    steals_df = pd.DataFrame(res)
    steals_df.to_csv(f'data/props/prizepicks/player_steals.csv', index=False)
    steals_df.head(10)

### underdog

In [12]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_steals')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_stl = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    player_team_abbr = player_df['TEAM_ABBREVIATION'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_steals(
        player_df, player_df_25, player_team, player_team_abbr, opp_team_id, opp_team,
        team_stats, league_avg_pace, league_avg_tov, home_flag, current_date, PLAYER, projectedStartingFive
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_stl % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_stl) + 1)
    elif target_stl % 1 == 0:
        prob_over_poisson = poisson.sf(target_stl, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_stl), int(target_stl) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_stl,
        'L-5': round(count_line_hits(player_df, target_stl, 'player_steals', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_stl, 'player_steals', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_stl, 'player_steals', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
    steals_df = pd.DataFrame(res)
    steals_df.to_csv(f'data/props/underdog/player_steals.csv', index=False)
    steals_df.head(10)

### prizepicks

In [13]:
## COMBO PROPS - All Categories

combo_categories = [
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
    'player_turnovers',
    'player_blocks_steals'
]

for category in combo_categories:
    print(f"\nProcessing {category}...")
    
    dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == category)]
    
    if dfs_data.empty:
        print(f"No data found for {category}")
        continue
    
    res = []
    PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))
    
    for _, row in PLAYERS.iterrows():
        PLAYER = row['NAME']
        target_line = row['LINE']

        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        if player_df.empty:
            continue

        res.append({
            'NAME': PLAYER,
            'LINE': target_line,
            'L-5': count_line_hits(player_df, target_line, category, [5])['L-5'],
            'L-10': count_line_hits(player_df, target_line, category, [10])['L-10'],
            'L-15': count_line_hits(player_df, target_line, category, [15])['L-15'],
        })
    
    if res:
        combo_df = pd.DataFrame(res).sort_values(by='L-5', ascending=False).reset_index(drop=True)
        combo_df.to_csv(f'data/props/prizepicks/{category}.csv', index=False)
        print(f"Saved {len(combo_df)} players for {category}")
        display(combo_df)
    else:
        print(f"No results for {category}")


Processing player_points_rebounds_assists...
Saved 106 players for player_points_rebounds_assists


,NAME,LINE,L-5,L-10,L-15
0,Jaylen Brown,41.5,1.0,0.8,0.67
1,Jose Alvarado,12.5,1.0,0.9,0.67
2,Victor Wembanyama,30.5,1.0,0.8,0.67
3,Jalen Johnson,42.5,1.0,0.7,0.53
4,Kevin Porter Jr.,33.0,0.8,0.4,0.27
...,...,...,...,...,...
101,Cade Cunningham,42.5,0.0,0.3,0.47
102,DeMar DeRozan,29.5,0.0,0.1,0.13
103,Keon Ellis,9.0,0.0,0.0,0.20
104,Kyle Kuzma,21.5,0.0,0.3,0.33



Processing player_points_rebounds...
Saved 115 players for player_points_rebounds


,NAME,LINE,L-5,L-10,L-15
0,Jalen Johnson,34.5,1.0,0.7,0.60
1,Jose Alvarado,9.5,1.0,0.9,0.73
2,Victor Wembanyama,26.5,1.0,0.9,0.73
3,Buddy Hield,11.5,1.0,0.7,0.60
4,Jordan Walsh,13.5,1.0,0.6,0.53
...,...,...,...,...,...
110,Nique Clifford,12.5,0.2,0.1,0.20
111,DeMar DeRozan,25.5,0.0,0.1,0.13
112,Kyle Kuzma,19.5,0.0,0.4,0.40
113,Kyshawn George,21.5,0.0,0.4,0.27



Processing player_points_assists...
Saved 104 players for player_points_assists


...

,NAME,LINE,L-5,L-10,L-15
0,Anthony Black,9.5,1.0,0.5,0.33
1,Jalen Johnson,18.5,1.0,0.8,0.60
2,Jaylen Brown,12.0,0.8,0.6,0.40
3,Maxime Raynaud,8.0,0.8,0.4,0.33
4,Jeremiah Fears,6.5,0.8,0.8,0.67
...,...,...,...,...,...
67,Shaedon Sharpe,8.0,0.2,0.3,0.33
68,Kevin Porter Jr.,11.5,0.2,0.1,0.07
69,Jalen Suggs,8.5,0.0,0.3,0.33
70,Myles Turner,7.0,0.0,0.3,0.40



Processing player_turnovers...
Saved 17 players for player_turnovers


,NAME,LINE,L-5,L-10,L-15
0,Stephen Curry,2.5,1.0,0.7,0.60
1,Deni Avdija,3.5,0.8,0.7,0.60
2,Jericho Sims,0.5,0.8,0.6,0.60
3,Moses Moody,0.5,0.8,0.7,0.67
4,Josh Okogie,0.5,0.8,0.5,0.47
5,Kyle Kuzma,1.5,0.8,0.7,0.53
6,Josh Minott,0.5,0.6,0.6,0.53
7,Chet Holmgren,1.5,0.6,0.5,0.53
8,Nicolas Batum,0.5,0.6,0.6,0.60
9,Jaden McDaniels,1.5,0.4,0.6,0.60



Processing player_blocks_steals...
Saved 13 players for player_blocks_steals


,NAME,LINE,L-5,L-10,L-15
0,Kevin Porter Jr.,1.5,1.0,0.5,0.33
1,Josh Okogie,1.5,0.8,0.6,0.60
2,Kris Dunn,1.5,0.8,0.5,0.47
3,Ryan Rollins,1.5,0.6,0.5,0.47
4,OG Anunoby,2.5,0.6,0.6,0.53
5,Victor Wembanyama,3.5,0.6,0.6,0.53
6,Anfernee Simons,0.5,0.4,0.4,0.33
7,Bobby Portis,0.5,0.4,0.5,0.40
8,James Harden,1.5,0.4,0.5,0.53
9,Kevin Durant,1.5,0.4,0.4,0.47


### underdog

In [14]:
## COMBO PROPS - All Categories

combo_categories = [
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
    'player_turnovers',
    'player_blocks_steals'
]

for category in combo_categories:
    print(f"\nProcessing {category}...")
    
    dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == category)]
    
    if dfs_data.empty:
        print(f"No data found for {category}")
        continue
    
    res = []
    PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))
    
    for _, row in PLAYERS.iterrows():
        PLAYER = row['NAME']
        target_line = row['LINE']

        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        if player_df.empty:
            continue

        res.append({
            'NAME': PLAYER,
            'LINE': target_line,
            'L-5': count_line_hits(player_df, target_line, category, [5])['L-5'],
            'L-10': count_line_hits(player_df, target_line, category, [10])['L-10'],
            'L-15': count_line_hits(player_df, target_line, category, [15])['L-15'],
        })
    
    if res:
        combo_df = pd.DataFrame(res).sort_values(by='L-5', ascending=False).reset_index(drop=True)
        combo_df.to_csv(f'data/props/underdog/{category}.csv', index=False)
        print(f"Saved {len(combo_df)} players for {category}")
        display(combo_df)
    else:
        print(f"No results for {category}")


Processing player_points_rebounds_assists...
Saved 97 players for player_points_rebounds_assists


,NAME,LINE,L-5,L-10,L-15
0,Jalen Johnson,42.5,1.0,0.7,0.53
1,Jaylen Brown,41.5,1.0,0.8,0.67
2,Victor Wembanyama,30.5,1.0,0.8,0.67
3,Ausar Thompson,19.5,0.8,0.5,0.60
4,Nickeil Alexander-Walker,28.5,0.8,0.7,0.60
...,...,...,...,...,...
92,VJ Edgecombe,21.5,0.2,0.5,0.47
93,Keegan Murray,25.5,0.2,0.3,0.20
94,Isaiah Jackson,12.5,0.2,0.3,0.47
95,Duop Reath,15.5,0.0,0.0,0.00



Processing player_points_rebounds...
Saved 51 players for player_points_rebounds


,NAME,LINE,L-5,L-10,L-15
0,Victor Wembanyama,27.5,1.0,0.8,0.67
1,Jalen Johnson,34.5,1.0,0.7,0.60
2,Saddiq Bey,22.5,1.0,0.9,0.67
3,Jaylen Brown,36.5,0.8,0.7,0.60
4,Trey Murphy III,27.5,0.8,0.6,0.67
5,Chet Holmgren,24.5,0.8,0.6,0.67
6,Josh Hart,22.5,0.8,0.7,0.60
7,Tyrese Maxey,32.5,0.8,0.5,0.60
8,Derrick White,23.5,0.8,0.6,0.53
9,Maxime Raynaud,18.5,0.8,0.4,0.27



Processing player_points_assists...
Saved 40 players for player_points_assists


...

,NAME,LINE,L-5,L-10,L-15
0,Anthony Black,9.5,1.0,0.5,0.33
1,Quentin Grimes,7.5,1.0,0.6,0.53
2,Trey Murphy III,9.5,0.8,0.5,0.60
3,Coby White,7.5,0.6,0.5,0.33
4,Steven Adams,8.5,0.6,0.7,0.73
5,Kon Knueppel,8.5,0.6,0.6,0.47
6,Ausar Thompson,8.5,0.6,0.5,0.53
7,James Harden,12.5,0.4,0.5,0.60
8,Stephen Curry,8.5,0.4,0.3,0.47
9,Josh Giddey,17.5,0.4,0.6,0.67



Processing player_turnovers...
Saved 11 players for player_turnovers


,NAME,LINE,L-5,L-10,L-15
0,Deni Avdija,3.5,0.8,0.7,0.60
1,Brandon Miller,2.5,0.6,0.3,0.20
2,Josh Giddey,3.5,0.6,0.5,0.53
3,Tyrese Maxey,2.5,0.6,0.6,0.60
4,Andrew Nembhard,2.5,0.6,0.5,0.47
5,Ryan Rollins,2.5,0.4,0.7,0.60
6,Kevin Durant,2.5,0.4,0.5,0.53
7,Shaedon Sharpe,2.5,0.4,0.5,0.40
8,Cade Cunningham,3.5,0.4,0.5,0.47
9,Jalen Johnson,3.5,0.4,0.6,0.60



Processing player_blocks_steals...
Saved 2 players for player_blocks_steals


,NAME,LINE,L-5,L-10,L-15
0,OG Anunoby,2.5,0.6,0.6,0.53
1,Victor Wembanyama,3.5,0.6,0.6,0.53
